# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [ ]:
%matplotlib ipympl

In [ ]:
import pandas as pd

from analysis.build_data import load_trace_df
from analysis.calibration import expected_calibration_error, kalshibench_metrics


In [ ]:
# one scoreable forecast per row, from every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence", run_name="sweep-20261106-1744")

In [ ]:
forecasts

## Expected calibration error

ECE (Naeini et al., 2015) is the average gap between confidence and accuracy, with forecasts binned by their stated confidence into $B$ bins:

$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{N}\,\bigl|\mathrm{acc}(B_b) - \mathrm{conf}(B_b)\bigr|$$

Here a forecast is the label the model predicts the market resolves to, `confidence` is the model's stated probability that the label is right, and a forecast is accurate when the label matches the market's `resolution` (ignoring case).

In [42]:
B = 10

In [43]:
confidence_bins = pd.IntervalIndex.from_breaks([b / B for b in range(B + 1)])

In [ ]:
forecasts["confidence_bin"] = pd.cut(
    forecasts["confidence"], confidence_bins, include_lowest=True
)

In [ ]:
ece = expected_calibration_error(forecasts)

In [ ]:
ece_by_model = forecasts.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

## KalshiBench table

Table 4 of [the paper](https://arxiv.org/pdf/2512.16030) reports, per model: accuracy, macro F1 and the F1 of the "yes" class, the Brier score of $P(\text{yes})$ against the outcome, the Brier skill score against a forecast of the base rate, $\mathrm{BSS} = 1 - \mathrm{BS} / \bar{y}(1 - \bar{y})$, and ECE and MCE (the worst bin's gap, $\max_b |\mathrm{acc}(B_b) - \mathrm{conf}(B_b)|$).

Here "yes" means the priced outcome occurs (`outcome_occurred`): the model said yes when its predicted resolution was that outcome, and its $P(\text{yes})$ is its confidence when it said yes and $1 -$ confidence otherwise.

In [ ]:
# the model called the priced outcome: its label was right on a market where that outcome occurred, or wrong where it didn't
forecasts["predicted_yes"] = forecasts["correct"] == forecasts[
    "outcome_occurred"
].astype(bool)

In [ ]:
forecasts["p_yes"] = forecasts["confidence"].where(
    forecasts["predicted_yes"], 1 - forecasts["confidence"]
)

In [ ]:
model_table = forecasts.groupby("model").apply(
    kalshibench_metrics, include_groups=False
)

In [50]:
model_table.round(3)

,Acc,F1,F1yes,Brier,BSS,ECE,MCE
model,,,,,,,
claude-fable-5-1,70.839,0.618,0.433,0.184,0.172,0.038,0.093
claude-opus-4-5,69.208,0.628,0.473,0.210,0.038,0.046,0.350
claude-opus-5-5,71.376,0.626,0.446,0.185,0.175,0.030,0.047


In [ ]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = forecasts.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)